# Chicago Taxi Pipeline — Silver Layer

## Purpose
Reads raw taxi data from the Bronze Delta table, applies
cleaning and transformation logic, and writes to the
Silver Delta table.

## Reads from
- `taxi_bronze` — raw ingested taxi data

## Writes to
- `taxi_silver` — cleaned and transformed taxi data

## Cleaning applied
- Casts trip_start and trip_end to proper timestamps
- Standardizes payment_type to title case
- Removes trips with zero miles or zero seconds
- Removes fares under $3.00

## Columns added
- `trip_minutes` — trip duration in minutes
- `fare_per_mile` — revenue efficiency per mile
- `tip_pct` — tip as a percentage of fare

## Notes
- Pipeline aborts if zero rows remain after cleaning
- 131 rows removed from 1000 in typical run (13%)

In [0]:
dbutils.widgets.text("run_date", "2025-05-24", "Run Date")

run_date = dbutils.widgets.get("run_date")

SOURCE_TABLE = "taxi_bronze"
TARGET_TABLE = "pipeline_silver"

print(f"Reading from: {SOURCE_TABLE}")
print(f"Writing to:   {TARGET_TABLE}")
print(f"Run date:     {run_date}")

In [0]:
from pyspark.sql.functions import col, round, when, lit, initcap, current_timestamp, to_timestamp
from pyspark.sql.types import DateType

In [0]:
df_bronze = spark.read.table(SOURCE_TABLE)
print(f"Rows read from Bronze: {df_bronze.count()}")
df_bronze.show(5)

In [0]:
df_silver = (df_bronze
    .withColumn("trip_start", to_timestamp(col("trip_start"), "yyyy-MM-dd'T'HH:mm:ss.SSS"))
    .withColumn("trip_end",   to_timestamp(col("trip_end"),   "yyyy-MM-dd'T'HH:mm:ss.SSS"))
    .withColumn("payment_type", initcap(col("payment_type")))
    .withColumn("trip_minutes", round(col("trip_seconds") / 60,1))
    .withColumn("fare_per_mile", round(col("fare") / col("trip_miles"),2))
    .withColumn("tip_pct", round((col("tips") / col("fare"))*100,1))
    .filter(col("trip_miles") > 0)
    .filter(col("trip_seconds") > 0)
    .filter(col("fare") >3 )
    )
print(f"Rows after cleaning: {df_silver.count()}")
df_silver.show(5)

In [0]:
row_count    = df_silver.count()

if row_count == 0:
    raise Exception("SILVER FAILED: No rows to write — aborting pipeline")

print("Validation passed — proceeding to write")
spark.sql("DROP TABLE IF EXISTS taxi_silver")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("taxi_silver")

print(f"taxi_silver written — {spark.read.table('taxi_silver').count()} rows")